# Pricing a European Call Option with Monte Carlo Simulation

## Motivation for this project

After studying Derivative Securities (FNCE30007) as a breadth, I wanted to move 
beyond the theory and actually build a pricing model. Black-Scholes gives a clean 
closed-form answer and equation for a European option, but most real-world derivatives don't have one. 
Monte Carlo simulation is the general tool used across the industry when no closed-form 
solution exists, so I wanted to understand it well enough to implement it myself.

This notebook builds a Monte Carlo pricer for a European call option, benchmarks it against the 
Black-Scholes price, and then improves it using a variance reduction technique called antithetic 
variates, which is something I have seen quite commonly in industry. 

## What is a European call option?

A European call option gives the holder the right (but not the obligation) to buy an underlying 
asset at a strike price $K$, at the expiry date $T$ in the future (unlike an American 
option, which can be exercised any time up to $T$). 

At expiry, the option is only worth exercising if the asset price $S_T$ has risen above the strike. 
The payoff is:

$$\text{Payoff} = \max(S_T - K, 0)$$

If $S_T > K$, you profit from the payoff. If $S_T \leq K$, the option expires worthless, with your downside limited to the price you paid for the option.

## The lognormal assumption

From the distribution of stock returns covered in lectures, the log stock price follows a 
normal distribution:

$$\ln S_T \sim \varphi\left[\ln S_0 + \left(\mu - \frac{\sigma^2}{2}\right)T,\ \sigma^2 T\right]$$

Since $\ln S_T$ is normally distributed, $S_T$ is lognormally distributed. This is an 
important modelling choice: a stock price can never go negative, but a normal distribution can. 
By modelling the log of the price as normal rather than the price itself, the model guarantees 
$S_T$ stays strictly positive, while still allowing for continuous, compounding growth, which is typical for a stock price.

Under risk-neutral pricing, the real-world drift $\mu$ is replaced with the risk-free rate $r$, given that all assets are assumed to grow at the risk-free rate on average:

$$\ln S_T \sim \varphi\left[\ln S_0 + \left(r - \frac{\sigma^2}{2}\right)T,\ \sigma^2 T\right]$$

Exponentiating both sides gives the simulation formula used below:

$$S_T = S_0 \exp\left[\left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z\right], \quad Z \sim N(0,1)$$

## Black-Scholes Model as a benchmark

The closed-form Black-Scholes price for a European call on a non-dividend paying stock is:

$$c = S_0 N(d_1) - Ke^{-rT}N(d_2)$$

where:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

## Why Monte Carlo simulation?

The fair price of a European call option is the discounted expected payoff under the 
risk-neutral measure:

$$c = e^{-rT}\,\mathbb{E}\left[\max(S_T - K, 0)\right]$$

Black-Scholes derives a closed-form solution for this expectation. But for many 
other derivatives (such as those with early exercise eg. American options), 
no such closed-form exists. Monte Carlo simulation approximates the 
expectation directly. Instead of solving for $\mathbb{E}[\max(S_T-K,0)]$ mathematically, we 
simulate many possible outcomes for $S_T$, compute the payoff for each one, and average them to get an approximate solution.

By the Law of Large Numbers, this average converges to the true expectation as we increase the number of 
simulations $N$. The steps are:

1. Simulate many possible terminal stock prices $S_T$, using the lognormal distribution assumption
2. Compute the option payoff $\max(S_T - K, 0)$ for each simulated price
3. Average the payoffs across all simulations
4. Discount the average back to today's value using $e^{-rT}$ (risk neutral valuation)

The Monte Carlo pricing method here is intentionally built on the same lognormal model used by 
Black-Scholes, so the two should converge to the same answer, which we will verify later on.

## How the simulation code works

As per the lecture slides, $\ln S_T$ is normally distributed with mean 
$\ln S_0 + (\mu - \sigma^2/2)T$ and variance $\sigma^2T$. To simulate a draw from this distribution, we write 
it as the mean plus the standard deviation multiplied by a standard normal random variable $Z$:

$$\ln S_T = \ln S_0 + \left(r - \frac{\sigma^2}{2}\right)T + \sigma\sqrt{T}\,Z, \qquad Z \sim N(0,1)$$

Exponentiating both sides 
gives the formula used in the code below.

We draw 10,000 independent values of $Z$ from the standard normal distribution, then plug each 
one into this formula to generate 10,000 corresponding terminal stock prices $S_T$. We print the first 5 simulated prices as a quick check that the output looks reasonable.

## Setting up the parameters

We define the inputs needed to price the option, which are: the current stock price, strike price, 
risk-free rate, volatility, time to maturity, and the number of Monte Carlo simulations to run. 
We also import the libraries needed: `numpy` for the random number generation and array 
operations, `matplotlib` for the convergence plot later on, and `scipy.stats.norm` for the 
normal cdf used in the Black-Scholes formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

S0 = 100        # current stock price
K = 100         # strike price
r = 0.05        # risk-free rate
sigma = 0.2     # volatility
T = 1.0         # time to maturity (years)
N = 100000     # number of simulations

## Calculating price using BSM formula

Before running the Monte Carlo simulation, we calculate the price using the 
Black-Scholes formula from lectures. This allows us to check our price derived from Monte Carlo simulations:

$$c = S_0 N(d_1) - Ke^{-rT}N(d_2)$$

where:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

We put this formula in a function, `black_scholes_call`.

In [ ]:
def black_scholes_call(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


bsm_price = black_scholes_call(S0, K, r, sigma, T)

## Plain Monte Carlo simulation

We now define a function which conducts the Monte Carlo simulation, `mc_call`. We simulate many terminal stock prices, compute the payoff for each, average 
them, and discount back to today.

We also calculate the standard error of our estimate, which measures how much uncertainty 
remains in our Monte Carlo price due to random sampling. A smaller standard error means a more 
precise estimate.

In [ ]:
def mc_call(S0, K, r, sigma, T, N):
    Z = np.random.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoff = np.maximum(ST - K, 0)
    price = np.exp(-r * T) * np.mean(payoff)
    std_error = np.exp(-r * T) * np.std(payoff) / np.sqrt(N)
    return price, std_error

## Monte Carlo with antithetic variates

This function implements the same simulation, but with antithetic variates for variance 
reduction. We draw half as many random values, then mirroring each one as its negative. By pairing $Z$ with $-Z$, the overshots and undershots in the 
simulated payoffs partially cancel out when averaged, reducing the variance of the estimate 
without introducing bias, since each individual $Z$ and $-Z$ is still a valid draw from the 
standard normal distribution.

In [ ]:
def mc_call_antithetic(S0, K, r, sigma, T, N):
    half_N = N // 2 # we will generate half as many random numbers since we are using antithetic variates
    Z = np.random.standard_normal(half_N)
    Z_anti = np.concatenate([Z, -Z])  # using antithetic variates, we now have N total samples

    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z_anti)
    payoff = np.maximum(ST - K, 0)
    price = np.exp(-r * T) * np.mean(payoff)
    std_error = np.exp(-r * T) * np.std(payoff) / np.sqrt(N)
    return price, std_error

## Running both methods and comparing results

We now call both `mc_call` and `mc_call_antithetic` with actual parameter values, and 
print all three: BSM, Monte Carlo, and antithetic Monte 
Carlo, along with their standard errors. We also calculate the variance reduction factor, which 
tells us how much more precise the antithetic method is compared to the original Monte Carlo method, for the 
same number of simulations.

`np.random.seed(1)` fixes the random number generator so the results are reproducible each time 
this code is run, rather than getting different random numbers on every execution.

In [ ]:
np.random.seed(1)
price_plain, se_plain = mc_call(S0, K, r, sigma, T, N)
price_anti, se_anti = mc_call_antithetic(S0, K, r, sigma, T, N)

print("Black-Scholes price:", bsm_price)
print("Monte Carlo price:", price_plain, "SE:", se_plain)
print("Antithetic Monte Carlo price:", price_anti, "SE:", se_anti)
print("Variance reduction factor:", (se_plain / se_anti) ** 2)

We get this output:

In [ ]:
Black-Scholes price: 10.450583572185565
Monte Carlo price: 10.501146042286969 | SE: 0.04664392559733537
Antithetic Monte Carlo price: 10.424228762496334 | SE: 0.04632267831286104
Variance reduction factor: 1.0139180729386266

## Conclusion
Both Monte Carlo estimates land close to the Black-Scholes price of 10.4506, confirming the 
simulation is correctly approximating the theoretical option value. The remaining difference in each 
case is simulation error from random sampling.

The variance reduction factor here is only about 1.01, which is almost no improvement 
from antithetic variates. This makes sense given the large number of simulations we did ($N = 10000$), both methods already 
have very small standard errors, leaving little room for antithetic pairing to reduce variance 
further. The benefit of antithetic variates is expected to be much more visible at smaller $N$, 
where plain Monte Carlo is noisier and there is more variance left to cancel out through pairing. 

## Next steps

1. Compare variance reduction at smaller $N$, where antithetic variates should show a more 
  pronounced benefit
2. Extend to American options, which allow early exercise and require a different simulation 
  approach 
3. Extend to Asian options, which depend on the average price over the option's life rather than 
  the terminal price alone. This is where Monte Carlo becomes essential for pricing, given that there is no known closed form solution